# Modes de vibration d'un réseau linéaire triatomique (1D)

Ce notebook calcule et visualise les relations de dispersion et les modes de vibration d'une chaîne triatomique 1D.
Nous utilisons `numpy` pour les calculs des valeurs propres de la matrice et `plotly` pour créer un graphique interactif.

In [39]:
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Label, Layout
from IPython.display import display
from matplotlib import pyplot as plt

# colors = [
#     ( 224,  32, 32),
#     (  32, 224, 32),
#     (  32,  32, 224),
#     (  32, 224, 224),
#     ( 224,  32, 224),
#     ( 224, 224,  32),
# ]
colors = ['blue', 'orange', 'green', 'purple', 'olive', 'cyan']  # Colors for plotting
n_colors = len(colors)

## Matrice dynamique et modes propres

La fonction `phi` construit la matrice dynamique pour un ensemble donné de constantes de raideur ($K_1, K_2, K_3$), de masses ($M_1, M_2, M_3$), de distances interatomiques ($a, b, c$) et de vecteur d'onde $q$.
La fonction `modes` calcule les valeurs propres (fréquences au carré $\omega^2$) et les vecteurs propres (déplacements atomiques) de la matrice dynamique.

In [40]:
def phi(K,M,d,q):
    return np.array([
        [(K[0]+K[2])/M[0], -K[0]/M[0]*np.exp(1j*q*d[0]), -K[2]/M[0]*np.exp(-1j*q*d[2])],
        [-K[0]/M[1]*np.exp(-1j*q*d[0]), (K[0]+K[1])/M[1], -K[1]/M[1]*np.exp(1j*q*d[1])],
        [-K[2]/M[2]*np.exp(1j*q*d[2]), -K[1]/M[2]*np.exp(-1j*q*d[1]), (K[1]+K[2])/M[2]]
    ])

def modes(K,M,d,q):
    w2, v = np.linalg.eig(phi(K,M,d,q))
    w = []
    for i in range(3):
        if np.imag(np.sqrt(w2[i])) < 1e-6:
            w.append(np.real(np.sqrt(w2[i])))
        elif np.real(np.sqrt(w2[i])) < 1e-6:
            w.append(-np.imag(np.sqrt(w2[i])))
        else:
            print("Error")
    order = np.argsort(w)
    return np.array(w)[order], ((v.T)[order]).T

## Fonction d'affichage (Plotting)

La fonction `func` ci-dessous est la routine principale d'affichage. Elle calcule les bandes pour toute une plage de valeurs de $q$ (de $0$ à $\pi/L$) et crée une disposition en sous-graphiques.
Le panneau de gauche montre la structure de bandes (relation de dispersion $\omega(q)$).
Les panneaux de droite montrent le déplacement physique des atomes dans l'espace réel pour les 3 différentes branches à un vecteur d'onde $q$ spécifique.

In [41]:
def func(K1, K2, K3, M1, M2, M3, a, b, c, q, n):
    K = np.array([K1, K2, K3])
    M = np.array([M1, M2, M3])
    d = np.array([a, b, c])
    L = np.sum(d)
    step = 1e-3
    qpts = np.arange(0, 1 + step, step)
    bands = [modes(K, M, d, np.pi / L * qi)[0] for qi in qpts]

    # Create figure with subplots
    fig, axes = plt.subplots(3, 2, figsize=(12, 8),
                             gridspec_kw={'height_ratios': [1, 1, 1],
                                         'width_ratios': [2, 1]})

    # Remove unused subplots
    fig.delaxes(axes[1, 0])
    fig.delaxes(axes[2, 0])

    # Plot dispersion relation
    ax_disp = axes[0, 0]
    for i in range(3):
        ax_disp.plot(qpts, [bands[j][i] for j in range(len(qpts))],
                    label=f'branche {i+1}', color=colors[i])
    ax_disp.set_xlabel('q (π/L)')
    ax_disp.set_ylabel('ω(q)')
    ax_disp.set_title('Bandes')
    ax_disp.legend()
    ax_disp.grid(True)

    # Calculate modes for selected q
    w, v = modes(K, M, d, np.pi / L * q)

    # Mark selected q on dispersion plot
    for i in range(3):
        ax_disp.scatter([q], [w[i]], color=colors[i], s=50)

    # Plot displacement patterns for each branch
    for i in range(3):
        ax = axes[i, 1]
        for j in range(3):
            x_pos = [k + np.sum(d[:j]) / L for k in range(-n, n + 1)]
            y_pos = [np.real(v[j, i]) * np.cos((k + np.sum(d[:j]) / L) * q * np.pi)
                    for k in range(-n, n + 1)]
            if i != 0:
                ax.scatter(x_pos, y_pos, color=colors[j + 3],
                          s=10 * (M[j])**(2) + 4, label=f'atome {j+1}' if i == 0 else None)
            else:
                ax.scatter(x_pos, y_pos, color=colors[j + 3],
                          s=10 * (M[j])**(2) + 4, label=f'atome {j+1}')
        ax.set_xlabel('x (L)')
        ax.set_ylabel('Déplacement')
        ax.set_title(f'Déplacement pour la branche {i+1}')
        ax.set_ylim(-1, 1)
        ax.grid(True)
        if i == 0:
            ax.legend()

    plt.tight_layout()
    plt.show()
    return

## Interface interactive

Ici, nous utilisons `ipywidgets` pour créer des curseurs pour les paramètres (constantes de raideur, masses, distances interatomiques, vecteur d'onde $q$, et le nombre de mailles à afficher).
La modification de ces curseurs mettra automatiquement à jour le graphique.

In [42]:
selector = widgets.Select(
    options=[],
    disabled=False,
    layout=Layout(width='150px', height='150px'))

K1 = widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
K2 = widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
K3 = widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
M1 = widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
M2 = widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
M3 = widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
a =  widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
b =  widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
c =  widgets.FloatSlider(value=0.5, min=0.1, max=2,  step=0.001, continuous_update=False, orientation='vertical')
q =  widgets.FloatSlider(value=0,   min=0.0, max=1,  step=0.001, continuous_update=False, orientation='vertical')
n =  widgets.IntSlider(value=2,     min=1,   max=5,  step=1,     continuous_update=False, orientation='vertical')

box1 = widgets.VBox([Label('K1'),K1])
box2 = widgets.VBox([Label('K2'),K2])
box3 = widgets.VBox([Label('K3'),K3])
box4 = widgets.VBox([Label('M1'),M1])
box5 = widgets.VBox([Label('M2'),M2])
box6 = widgets.VBox([Label('M3'),M3])
box7 = widgets.VBox([Label('a'),a])
box8 = widgets.VBox([Label('b'),b])
box9 = widgets.VBox([Label('c'),c])
box10 = widgets.VBox([Label('q (π/L)'),q])
box11 = widgets.VBox([Label('n_cell'),n])

ui = widgets.HBox([box1, box2, box3, box4, box5, box6, box7, box8, box9, box10, box11])

out = widgets.interactive_output(
    func,{
        'K1': K1, 'K2': K2, 'K3': K3,
        'M1': M1, 'M2': M2, 'M3': M3,
        'a':  a,  'b':  b,  'c':  c,
        'q':  q,  'n':  n
    }
)

display(ui,out)


Output()